# Цифровые технологии в профессиональной деятельности
## Раздел 1. Текстовые данные
### Семинар 1. Количественные методы и контент-анализ
### Материал для самостоятельного изучения: наивная токенизация

В рамках данного материала мы рассмотрим, как работает наивная токенизация текста.

Материалом послужит текст первой главы *Алисы в Стране чудес* в переводе Бориса Заходера.

### 0. Загрузка файла

In [3]:
import requests

In [4]:
url = r"https://raw.githubusercontent.com/always1ready/dh-tools/refs/heads/main/data/01_seminar/alice_in_wonderland_ch1.txt"

response = requests.get(url)

alice_text = response.text

print(len(alice_text))
print(alice_text[:323])

12858
Алиса сидела со старшей сестрой на берегу и маялась делать ей было
совершенно нечего, а сидеть без дела, сами знаете, дело нелегкое; раз-другой
она, правда, сунула нос в книгу, которую сестра читала, но там не оказалось
ни картинок, ни стишков. Кому нужны книжки без картинок.- или хоть стишков,
не понимаю! - думала Ал


Посимвольный объём текста сообщает нам некоторую информацию о его величине, но в действительности среди всех символов этого файла значительная часть пробельные отступы. Привычнее и понятнее получать какую-то количественную информацию о тексте на основе подсчёта слов. Попробуем посчитать слова с помощью встроенных функций Python.

### 1. Предобработка текста: токенизация по словам

Для того чтобы проанализировать содержание, текст нужно разбить на отдельные слова — **токены**.

In [5]:
# токенами могут быть разные части текста, например, фрагменты между запятыми

comma_tokens = alice_text.split(",")
print(comma_tokens[:5])

['Алиса сидела со старшей сестрой на берегу и маялась делать ей было\r\nсовершенно нечего', ' а сидеть без дела', ' сами знаете', ' дело нелегкое; раз-другой\r\nона', ' правда']


In [6]:
# или между точками

point_tokens = alice_text.split(".")
print(point_tokens[:5])

['Алиса сидела со старшей сестрой на берегу и маялась делать ей было\r\nсовершенно нечего, а сидеть без дела, сами знаете, дело нелегкое; раз-другой\r\nона, правда, сунула нос в книгу, которую сестра читала, но там не оказалось\r\nни картинок, ни стишков', ' Кому нужны книжки без картинок', '- или хоть стишков,\r\nне понимаю! - думала Алиса', '\r\n     С горя она начала подумывать (правда, сейчас это тоже было дело не\r\nиз легких - от жары ее совсем разморило), что, конечно, неплохо бы сплести\r\nвенок из маргариток, но плохо то, что тогда нужно подниматься и идти\r\nсобирать эти маргаритки, как вдруг', '']


In [7]:
# но человеку привычнее воспринимать информацию через минимальные самостоятельные единицы - слова

naive_tokens = alice_text.split()

print("Как выглядят наши токены:")
print(naive_tokens[40:50])

Как выглядят наши токены:
['Кому', 'нужны', 'книжки', 'без', 'картинок.-', 'или', 'хоть', 'стишков,', 'не', 'понимаю!']


Попробуем посчитать частотность токенов в тексте, для этого воспользуемся классом `Counter()` из встроенного модуля `collections`. Он работает как «умный» словарь, который сам считает количество повторений каждого элемента.

In [8]:
# Как работает Counter()
from collections import Counter

# Представим, что это результат нашей токенизации
example_tokens = ['Алиса', 'кролик', 'Кролик', 'Алиса', 'Выпей меня', 'Чеширский', 'Кролик']

# Создаем объект Counter
word_counts = Counter(example_tokens)

print("Весь частотный словарь:")
print(word_counts)

print(f"\nСколько раз встречается 'Алиса'?: {word_counts['Алиса']}")

# Метод .most_common(n) возвращает n самых частотных слов
print("\nТоп-2 самых частых слова:")
print(word_counts.most_common(2))

Весь частотный словарь:
Counter({'Алиса': 2, 'Кролик': 2, 'кролик': 1, 'Выпей меня': 1, 'Чеширский': 1})

Сколько раз встречается 'Алиса'?: 2

Топ-2 самых частых слова:
[('Алиса', 2), ('Кролик', 2)]


Теперь применим `Counter()` ко всему тексту.

In [9]:
word_counts = Counter(naive_tokens)

print("Всего токенов:")
print(len(naive_tokens))

print("\nСамые частотные токены:")
print(word_counts.most_common(10))

print("\nФрагмент частотного словаря:")
print(f"Слово 'Алиса': {word_counts.get('Алиса')}")
print(f"Слово 'Алиса.': {word_counts.get('Алиса.')}")
print(f"Слово 'Алиса,': {word_counts.get('Алиса,')}")

print(f"\nСлово 'кролик': {word_counts.get('кролик')}")
print(f"Слово 'Кролик': {word_counts.get('Кролик')}")

Всего токенов:
2022

Самые частотные токены:
[('и', 67), ('не', 48), ('-', 38), ('что', 38), ('она', 34), ('на', 32), ('Алиса', 30), ('в', 27), ('как', 25), ('я', 19)]

Фрагмент частотного словаря:
Слово 'Алиса': 30
Слово 'Алиса.': 1
Слово 'Алиса,': 1

Слово 'кролик': 1
Слово 'Кролик': 5


Стандартный строковый метод `.split()` работает грубо, оставляя знаки препинания приклеенными к словам. Для того чтобы очистить текст от «пунктуационного шума», в Python чаще всего используют встроенный модуль `string`. Он содержит готовую строку `punctuation`, в которой собраны все основные спецсимволы (тире, кавычки, точки, скобки и т.д.).

In [10]:
import string

punctuation = string.punctuation + '«»—' # к сож, некоторые символы, часто используемые в русской типографике, отсутствуют

print(f"Стандартные знаки препинания: {punctuation}")

Стандартные знаки препинания: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~«»—


In [11]:
clean_tokens = []

for word in naive_tokens:
    # Метод .strip() удаляет указанные символы только с краев слова
    clean_word = word.strip(punctuation)
    
    # Добавляем в список только если после очистки что-то осталось 
    # (чтобы не забивать список пустыми строками от одиночных тире)
    if clean_word:
        clean_tokens.append(clean_word.lower()) # заодно все токены приведём в нижний регистр

word_counts = Counter(clean_tokens)

print("\nТокены после базовой очистки:")
print(clean_tokens[:10])

print("\nВсего токенов:")
print(len(clean_tokens))

print("\nСамые частотные токены:")
print(word_counts.most_common(10))

print("\nФрагмент частотного словаря:")
print(f"Слово 'алиса': {word_counts.get('алиса')}")
print(f"Слово 'Алиса': {word_counts.get('Алиса')}")
print(f"Слово 'Алиса.': {word_counts.get('Алиса.')}")
print(f"Слово 'Алиса,': {word_counts.get('Алиса,')}")

print(f"\nСлово 'кролик': {word_counts.get('кролик')}")
print(f"Слово 'Кролик': {word_counts.get('Кролик')}")


Токены после базовой очистки:
['алиса', 'сидела', 'со', 'старшей', 'сестрой', 'на', 'берегу', 'и', 'маялась', 'делать']

Всего токенов:
1983

Самые частотные токены:
[('и', 77), ('не', 51), ('она', 48), ('что', 41), ('алиса', 37), ('на', 35), ('как', 32), ('в', 28), ('а', 21), ('я', 21)]

Фрагмент частотного словаря:
Слово 'алиса': 37
Слово 'Алиса': None
Слово 'Алиса.': None
Слово 'Алиса,': None

Слово 'кролик': 7
Слово 'Кролик': None


Получилось уже значительно лучше, однако результат всё ещё не идеален, вот примеры:

In [12]:
print(clean_tokens[111:128]) # дефисы не требующие разделения
print(clean_tokens[273:286]) # дефисы при редупликации (повторе слов), необходимо разделить

['алиса-то', 'не', 'так', 'уж', 'удивилась', 'даже', 'когда', 'услыхала', 'что', 'кролик', 'сказал', 'а', 'сказал', 'он', 'ай-ай-ай', 'я', 'опаздываю']
['алиса', 'ахнуть', 'не', 'успела', 'как', 'полетела-полетела', 'вниз', 'в', 'какой-то', 'очень', 'очень', 'глубокий', 'колодец']


Текст пронизан такими проблемами сплошь и рядом, а анализируя философский текст или исторический источник, а тем более художественный или искусствоведческий текст проблем токенизации может оказаться сильно больше. 

К тому же сам алгоритм очень жадный и ресурсоёмкий. Для обработки 2 тысяч токенов сойдёт, но когда в тексте или в целом корпусе миллионы токенов, процесс сильно замедлится.

На семинаре мы попробуем разобраться с этой проблемой, поработаем с NLP-библиотеками, изучим или повторим регулярные выражения.